# Esquema: regresión + sklearn + PyTorch (MVP)

CSV con faltantes → **tratamiento manual** en pandas → split → varios modelos **sklearn** en `Pipeline` → **MLP en PyTorch** → comparación en val (elegir modelo) y test (evaluación final).

| Paso | Contenido |
|------|-----------|
| 1–5 | CSV (`data/datos_casas.csv`), tipos, imputación, dummies, split train/val/test |
| 6 | Entrenar modelos **sklearn** (`build_models()`, solo fit) |
| 7 | Entrenar red **PyTorch** (`HousePriceNet`) |
| 8 | **Análisis comparativo** (tabla val/test, elegir ganador) |

> Ejecuta el notebook desde `13-esquemas-sklearn-pytorch/`.

Solo **`fit` en train**; predicciones y métricas van en el apartado de análisis.

Predicciones en **val** y **test**, tabla comparativa y elección del ganador por **val**.


## 1. Importar CSV y revisar datos

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("data/datos_casas.csv")
print("Tipos:\n", df.dtypes)
print("\nFaltantes:\n", df.isna().sum())
display(df)


Tipos:
 Metros_Cuadrados    float64
Zona                 object
Precio                int64
dtype: object

Faltantes:
 Metros_Cuadrados    1
Zona                1
Precio              0
dtype: int64


,Metros_Cuadrados,Zona,Precio
0,45.0,Centro,125000
1,52.0,Periferia,138000
2,NaN,Centro,132000
3,61.0,Periferia,155000
4,70.0,Centro,172000
5,85.0,NaN,195000
6,95.0,Periferia,210000
7,110.0,Centro,245000
8,130.0,Periferia,285000
9,150.0,Centro,320000


## 2. Target numérico (`Precio`)

In [9]:
df = df.dropna(subset=["Precio"]).copy()
y = df["Precio"]


## 3–4. Features (manual)

In [10]:
metros = df["Metros_Cuadrados"].astype(float)
X_num = pd.DataFrame({"Metros_Cuadrados": metros.fillna(metros.median())})
zona = df["Zona"].fillna("Desconocida").astype(str)
X_cat = pd.get_dummies(zona, prefix="Zona", dtype=float)
X = pd.concat([X_num, X_cat], axis=1)


## 5. Split train / val / test


In [11]:
def split_train_val_test(X, y, test_size, val_size, random_state, stratify=False):
    """Divide en train, validación y test (dos llamadas a train_test_split).

    - test_size: fracción del total para test (hold-out final).
    - val_size: fracción de train+val → validación.
    Con test_size=0.2 y val_size=0.25 → ~60 % train, ~20 % val, ~20 % test.
    """
    kw = dict(test_size=test_size, random_state=random_state)
    if stratify:
        kw["stratify"] = y
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, **kw)
    kw2 = dict(test_size=val_size, random_state=random_state)
    if stratify:
        kw2["stratify"] = y_tv
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, **kw2)
    return X_train, X_val, X_test, y_train, y_val, y_test


RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.25

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(f"Tamaños → train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")


Tamaños → train: 6 | val: 2 | test: 3


## 6. Entrenar modelos sklearn

**Fit** solo en train; métricas en el paso 8.

Cada `Pipeline` define **por separado**:
- **`transformacion`**: `SimpleImputer` (imputación de faltantes; complementa pandas pasos 3–4).
- **`estandarizado`**: `StandardScaler` (media 0, desv. 1).
- **`modelo`**: estimador de `build_models()`.


In [12]:
def build_models():
    """Comenta entradas del dict para excluir modelos."""
    from sklearn.ensemble import (
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.linear_model import Lasso, LinearRegression, Ridge
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.tree import DecisionTreeRegressor
    from xgboost import XGBRegressor
    from catboost import CatBoostRegressor

    return {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=5000),
        "DecisionTree": DecisionTreeRegressor(
            criterion="squared_error",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            criterion="squared_error",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=1.0,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        "KNN": KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
        "XGBoost": XGBRegressor(
            random_state=RANDOM_STATE, verbosity=0, n_estimators=100, n_jobs=-1
        ),
        "CatBoost": CatBoostRegressor(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }


RANDOM_STATE = 42
MODELS = build_models()

def build_sklearn_pipeline(modelo):
    """Pipeline con transformación y estandarizado como pasos separados."""
    return Pipeline([
        ("transformacion", SimpleImputer(strategy="median")),  # imputación (transformación)
        ("estandarizado", StandardScaler()),                   # media 0, desv. 1
        ("modelo", modelo),
    ])


pipelines = {}
for nombre, modelo in MODELS.items():
    pipe = build_sklearn_pipeline(modelo)
    pipe.fit(X_train, y_train)
    pipelines[nombre] = pipe



## 7. Red neuronal (PyTorch)

Misma **X** / **y** que sklearn. Escalado solo en train. **MLP tabular**: capas densas + `BatchNorm1d` + ReLU + `Dropout` + salida lineal.

> Referencia: [12-pytorch/00-pytorch-cheat-sheet.ipynb](../12-pytorch/00-pytorch-cheat-sheet.ipynb)


In [13]:
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

# Transformación (imputación; stats solo de train)
imputer_nn = SimpleImputer(strategy="median")
X_tr = imputer_nn.fit_transform(X_train)
X_va = imputer_nn.transform(X_val)
X_te = imputer_nn.transform(X_test)

# Estandarizado (media/std solo de train)
scaler_nn = StandardScaler()
X_tr = scaler_nn.fit_transform(X_tr)
X_va = scaler_nn.transform(X_va)
X_te = scaler_nn.transform(X_te)

n_in = X_tr.shape[1]


# MLP para regresión: capas densas + BatchNorm + ReLU + Dropout; salida lineal (un solo valor).
class HousePriceNet(nn.Module):
    def __init__(self, n_features: int, dropout_rate: float = 0.3):
        super().__init__()
        # Bloques de ancho decreciente: patrón habitual en tabular (ancho → estrecho).
        self.network = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),  # Normaliza activaciones por batch.
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),  # Menos dropout cerca de la salida.
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 1),  # Sin sigmoid: valor continuo (objetivo escalado).
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)  # x: (batch, n_features) → (batch, 1)


modelo_nn = HousePriceNet(n_in).to(device)
criterio = nn.MSELoss()
optimizador = torch.optim.Adam(modelo_nn.parameters(), lr=1e-2)

# Escalar también y (solo stats de train) — estabiliza el MLP con pocos datos
y_mean, y_std = float(y_train.mean()), float(y_train.std()) + 1e-8
y_tr_norm = ((y_train - y_mean) / y_std).values.reshape(-1, 1)

X_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_t = torch.tensor(y_tr_norm, dtype=torch.float32, device=device)

EPOCHS = 400
for ep in range(EPOCHS):
    modelo_nn.train()
    optimizador.zero_grad()
    perdida = criterio(modelo_nn(X_t), y_t)
    perdida.backward()
    optimizador.step()



## 8. Análisis comparativo (sklearn + PyTorch)

Elige el mejor modelo por **R² en val**; tabla con métricas en val y test.

Predicciones en **val** y **test**, tabla comparativa y elección del ganador por **val**.


In [14]:
filas = []
for nombre, pipe in pipelines.items():
    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)
    filas.append(
        {
            "modelo": nombre,
            "R2_val": r2_score(y_val, pred_val),
            "R2_test": r2_score(y_test, pred_test),
            "MSE_val": mean_squared_error(y_val, pred_val),
            "MSE_test": mean_squared_error(y_test, pred_test),
        }
    )

modelo_nn.eval()
with torch.no_grad():
    pred_norm_val = modelo_nn(torch.tensor(X_va, dtype=torch.float32, device=device))
    pred_norm_te = modelo_nn(torch.tensor(X_te, dtype=torch.float32, device=device))
    pred_nn_val = pred_norm_val.cpu().numpy().ravel() * y_std + y_mean
    pred_nn = pred_norm_te.cpu().numpy().ravel() * y_std + y_mean

r2_nn_val = r2_score(y_val, pred_nn_val)
r2_nn_test = r2_score(y_test, pred_nn)
mse_nn_val = mean_squared_error(y_val, pred_nn_val)
mse_nn_test = mean_squared_error(y_test, pred_nn)

comparacion = pd.concat(
    [
        pd.DataFrame(filas),
        pd.DataFrame(
            [{
                "modelo": "PyTorch_MLP",
                "R2_val": r2_nn_val,
                "R2_test": r2_nn_test,
                "MSE_val": mse_nn_val,
                "MSE_test": mse_nn_test,
            }]
        ),
    ],
    ignore_index=True,
).sort_values("R2_val", ascending=False)

display(comparacion.round(4))

mejor = comparacion.iloc[0]
print(f"\nMejor modelo en val: {mejor['modelo']} (R²_val = {mejor['R2_val']:.4f})")
print(f"  R² en test = {mejor['R2_test']:.4f} | MSE en test = {mejor['MSE_test']:,.0f}")



,modelo,R2_val,R2_test,MSE_val,MSE_test
8,XGBoost,0.5575,0.9015,1.412500e+09,6.410009e+08
4,RandomForest,0.3177,0.9371,2.178004e+09,4.092455e+08
1,Ridge,0.1630,0.9776,2.671839e+09,1.455775e+08
10,PyTorch_MLP,0.1073,0.3207,2.849581e+09,4.419143e+09
2,Lasso,0.0474,0.9987,3.040866e+09,8.391811e+06
0,LinearRegression,0.0473,0.9983,3.041103e+09,1.113918e+07
7,KNN,-0.0038,0.4449,3.204500e+09,3.611387e+09
9,CatBoost,-0.0310,0.2905,3.291363e+09,4.615718e+09
5,GradientBoosting,-0.0359,0.9158,3.306905e+09,5.480485e+08
3,DecisionTree,-0.1448,0.8125,3.654500e+09,1.219667e+09



Mejor modelo en val: XGBoost (R²_val = 0.5575)
  R² en test = 0.9015 | MSE en test = 641,000,896
